## 10_ Structured_+ NLP_+ External_Context(Matrix D)

Explainable AI Credit Risk Decision Platform: Integrating Structured Borrower Data, NLP-Driven Text Intelligence, and Macroeconomic Indicators for Transparent Lending Decisions.

#### Reason:
Translating unstructured natural language processing features into an organized, tabular schema that integrates external contextual parameters.

In [148]:
# Standard Libraries
# -----------------------------

import os
from pathlib import Path
from datetime import datetime
import json
import logging
import warnings

# Data Processing
# -----------------------------

import numpy as np
import pandas as pd

# Sparse Matrix Processing
# -----------------------------

from scipy.sparse import load_npz, save_npz, hstack, csr_matrix

# Visualisation
# -----------------------------

import matplotlib.pyplot as plt


# Ignore Warnings
# -----------------------------

warnings.filterwarnings("ignore")

# RANDOM STATE
# -----------------------------

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

In [173]:
# PROJECT DIRECTORIES
# -------------------------------------------

PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"

FEATURE_STORE_DIR = DATA_DIR / "feature_store"

FEATURE_REPORT_DIR = REPORT_DIR / "feature_engineering"

REPORT_DIR = PROJECT_ROOT / "reports"

MODEL_DIR = PROJECT_ROOT / "models"

LOG_DIR = PROJECT_ROOT / "logs"

# Create folders if missing

for folder in [FEATURE_STORE_DIR,REPORT_DIR,MODEL_DIR,LOG_DIR]:
    
    folder.mkdir(parents=True,exist_ok=True)

In [109]:
# LOGGING CONFIGURATION
# ________________________________________

logging.basicConfig(

    filename=LOG_DIR / "matrix_D_pipeline.log",

    level=logging.INFO,

    format="%(asctime)s | %(levelname)s | %(message)s")

logger = logging.getLogger()

logger.info("Matrix D pipeline started")

In [110]:
# OUTPUT FILES
# __________________________________________________

MATRIX_D_DIR = FEATURE_STORE_DIR / "matrix_d"

MATRIX_D_DIR.mkdir(parents=True, exist_ok=True)

In [111]:
# MATRIX C FILE LOCATIONS
# ________________________________

MATRIX_C_TRAIN_PATH = (FEATURE_STORE_DIR /"matrix_C_train.npz")

MATRIX_C_TEST_PATH = (FEATURE_STORE_DIR /"matrix_C_test.npz")

# MACROECONOMIC FEATURE STORE LOCATION
# ______________________________________________

MACRO_FEATURE_PATH = (FEATURE_STORE_DIR /"macroeconomic_feature_store.parquet")


In [112]:
# LOAD MATRIX C FILES 
# ___________________________________________________

logger.info("Loading Matrix C...")

MATRIX_C_TRAIN = load_npz(MATRIX_C_TRAIN_PATH)

MATRIX_C_TEST = load_npz(MATRIX_C_TEST_PATH)

logger.info("Matrix C loaded successfully.")

print("Training Matrix Shape :", MATRIX_C_TRAIN.shape)

print("Testing Matrix Shape  :", MATRIX_C_TEST.shape)

Training Matrix Shape : (400000, 70)
Testing Matrix Shape  : (100000, 70)


In [113]:
# DATASET METADATA
# __________________________________________

TARGET_COLUMN = "loan_status"


DATE_COLUMN = "issue_d"


TIME_KEY = "YearMonth"

In [114]:
# LOAD MATRIX C MAPPING
#_________________________________________

train_mapping = pd.read_parquet(FEATURE_STORE_DIR /"matrix_C_training_mapping.parquet")

test_mapping = pd.read_parquet(FEATURE_STORE_DIR /"matrix_C_testing_mapping.parquet")

train_mapping.head()

test_mapping.head()

,row_index,issue_d,YearMonth,loan_status
0,0,2015-11-01,2015-11,Charged Off
1,1,2015-03-01,2015-03,Fully Paid
2,2,2015-10-01,2015-10,Fully Paid
3,3,2015-10-01,2015-10,Fully Paid
4,4,2018-03-01,2018-03,Current


In [115]:
# SECTION 2: LOAD MACROECONOMIC FEATURE STORE
# ___________________________________________________

macro_store = pd.read_parquet(FEATURE_STORE_DIR /"macroeconomic_feature_store.parquet")

macro_store.head()

macro_store.shape

(144, 25)

In [116]:
# VALIDATE YEARMONTH ALIGNMENT
# _______________________________________

print("Matrix C months:")

print(train_mapping["YearMonth"].dropna().unique()[:10])

print("\nMacro Store months:")

print(macro_store["YearMonth"].unique()[:10])

Matrix C months:
['2015-01' '2015-12' '2015-05' '2018-03' '2015-09' '2015-03' '2018-01'
 '2015-10' '2015-07' '2015-04']

Macro Store months:
['2007-01' '2007-02' '2007-03' '2007-04' '2007-05' '2007-06' '2007-07'
 '2007-08' '2007-09' '2007-10']


In [117]:
# VALIDATE MATRIX C MAPPING
# __________________________________________________________

assert MATRIX_C_TRAIN.shape[0] == len(train_mapping), \
    "Training Matrix C and mapping row counts do not match."

assert MATRIX_C_TEST.shape[0] == len(test_mapping), \
    "Testing Matrix C and mapping row counts do not match."

logger.info("Matrix C and mapping are perfectly aligned.")

print("Training:", MATRIX_C_TRAIN.shape[0], "rows")

print("Testing :", MATRIX_C_TEST.shape[0], "rows")

Training: 400000 rows
Testing : 100000 rows


In [118]:
# HANDLE MISSING TEMPORAL INFORMATION
# ______________________________________________

logger.info("Handling missing temporal information...")

for name, df in {"Training": train_mapping,"Testing": test_mapping}.items():

    # Detect invalid issue_d values
    # --------------------------------------------------------

    invalid_dates = ((df["issue_d"].isna()) |
                     
                     (df["issue_d"] <= pd.Timestamp("1900-01-01")))


    # Replace invalid dates with NaT

    df.loc[invalid_dates,"issue_d"] = pd.NaT

    # Create data availability flags
    # --------------------------------------------------------

    df["MACRO_DATA_AVAILABLE"] = (df["YearMonth"].notna().astype(int))
    
    df["DATE_AVAILABLE"] = (df["issue_d"].notna().astype(int))

    # Replace missing YearMonth
    # --------------------------------------------------------

    df["YearMonth"] = (df["YearMonth"].fillna("UNKNOWN"))
    
    logger.info(f"{name}: "f"{invalid_dates.sum()} invalid dates handled.")

logger.info("Temporal missing value handling completed.")

In [119]:
# CREATE UNKNOWN MACRO PROFILE
# ___________________________________

logger.info("Creating UNKNOWN macro profile...")

if "UNKNOWN" not in macro_store["YearMonth"].values:

    macro_unknown = {"YearMonth": "UNKNOWN"}

    for col in macro_store.columns:
        if col != "YearMonth":
            macro_unknown[col] = 0

    macro_store = pd.concat(
        [
            macro_store,
            pd.DataFrame([macro_unknown])
        ],
        ignore_index=True)

logger.info("UNKNOWN macro profile added.")

In [120]:
# MERGE MATRIX C MAPPING WITH MACRO FEATURE STORE
# _________________________________________________________

logger.info("Merging Matrix C Mapping with Macroeconomic Feature Store...")

# Merge Training Mapping
# ___________________________________________________

train_mapping_macro = train_mapping.merge(macro_store,
                                          
                                          on="YearMonth",
                                          
                                          how="left",
                                          
                                          validate="many_to_one")

logger.info("Training mapping merged successfully.")

# Merge Testing Mapping
# ______________________________________________________

test_mapping_macro = test_mapping.merge(macro_store,
                                        
                                        on="YearMonth",
                                        
                                        how="left",
                                        
                                        validate="many_to_one")

logger.info("Testing mapping merged successfully.")

# Display Results
# _______________________________________________

print("TRAINING MAPPING + MACRO FEATURES")

display(train_mapping_macro.head())

print("\nShape:", train_mapping_macro.shape)

print("\n")

print("=" * 60)
print("TESTING MAPPING + MACRO FEATURES")
print("=" * 60)

display(test_mapping_macro.head())

print("\nShape:", test_mapping_macro.shape)

logger.info("Matrix C Mapping successfully enriched with macroeconomic features.")

TRAINING MAPPING + MACRO FEATURES


,row_index,issue_d,YearMonth,loan_status,MACRO_DATA_AVAILABLE,DATE_AVAILABLE,DATE,Year,Month,FEDFUNDS,UNRATE,DGS10,UMCSENT,FEDFUNDS_MoM,CPIAUCSL_YoY,CPIAUCSL_3M_Momentum,GDPC1_Growth,YIELD_CURVE,Articles,LOAN,MORTGAGE,FEDERAL_RESERVE,ECONOMY,Average_Tone,Average_Tone_LAG3,UNRATE_CHANGE,Average_Tone_CHANGE,ECONOMIC_STRESS_INDEX,RECESSION_FLAG,HIGH_RATE_ENVIRONMENT
0,0,2015-01-01,2015-01,Current,1,1,2015-01-01 00:00:00,2015,1,0.11,5.7,1.881500,98.1,-0.01,-0.229931,-1.130017,0.900485,1.330000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.1,0.000000,5.580069,0,0
1,1,2015-12-01,2015-12,Fully Paid,1,1,2015-12-01 00:00:00,2015,12,0.24,5.0,2.242727,92.6,0.12,0.638725,0.110738,0.000000,1.260000,69756.0,4.0,0.0,272.0,8378.0,-0.990107,-0.926054,-0.1,0.004727,5.878725,0,1
2,2,2015-05-01,2015-05,Fully Paid,1,1,2015-05-01 00:00:00,2015,5,0.12,5.6,2.197500,90.7,0.00,0.035033,0.704932,0.000000,1.588500,68502.0,1.0,0.0,0.0,29.0,-1.143442,-1.217589,0.2,0.055694,5.755033,0,0
3,3,2018-03-01,2018-03,Current,1,1,2018-03-01 00:00:00,2018,3,1.51,4.0,2.842381,101.4,0.09,2.330950,0.715078,0.000000,0.566667,71380.0,0.0,0.0,251.0,16083.0,-0.992708,-0.900501,-0.1,-0.095727,7.840950,0,1
4,4,2015-09-01,2015-09,Fully Paid,1,1,2015-09-01 00:00:00,2015,9,0.14,5.0,2.172857,87.2,0.00,0.008843,-0.066903,0.000000,1.459524,79610.0,6.0,0.0,0.0,24.0,-0.926054,-1.290044,-0.1,0.114194,5.148843,0,0



Shape: (400000, 30)


TESTING MAPPING + MACRO FEATURES


,row_index,issue_d,YearMonth,loan_status,MACRO_DATA_AVAILABLE,DATE_AVAILABLE,DATE,Year,Month,FEDFUNDS,UNRATE,DGS10,UMCSENT,FEDFUNDS_MoM,CPIAUCSL_YoY,CPIAUCSL_3M_Momentum,GDPC1_Growth,YIELD_CURVE,Articles,LOAN,MORTGAGE,FEDERAL_RESERVE,ECONOMY,Average_Tone,Average_Tone_LAG3,UNRATE_CHANGE,Average_Tone_CHANGE,ECONOMIC_STRESS_INDEX,RECESSION_FLAG,HIGH_RATE_ENVIRONMENT
0,0,2015-11-01,2015-11,Charged Off,1,1,2015-11-01 00:00:00,2015,11,0.12,5.1,2.263158,91.3,0.00,0.436318,-0.006722,0.000000,1.378421,62568.0,1.0,0.0,0.0,24.0,-0.994834,-1.040248,0.1,-0.020942,5.656318,0,0
1,1,2015-03-01,2015-03,Fully Paid,1,1,2015-03-01 00:00:00,2015,3,0.11,5.4,2.042727,93.0,0.00,-0.022031,-0.116824,0.000000,1.402273,71368.0,8.0,4.0,0.0,25.0,-1.129146,0.000000,-0.1,0.088443,5.487969,0,0
2,2,2015-10-01,2015-10,Fully Paid,1,1,2015-10-01 00:00:00,2015,10,0.12,5.0,2.070000,90.0,-0.02,0.127617,-0.126453,0.184479,1.425238,83332.0,10.0,1.0,0.0,29.0,-0.973892,-0.911621,0.0,-0.047839,5.247617,0,0
3,3,2015-10-01,2015-10,Fully Paid,1,1,2015-10-01 00:00:00,2015,10,0.12,5.0,2.070000,90.0,-0.02,0.127617,-0.126453,0.184479,1.425238,83332.0,10.0,1.0,0.0,29.0,-0.973892,-0.911621,0.0,-0.047839,5.247617,0,0
4,4,2018-03-01,2018-03,Current,1,1,2018-03-01 00:00:00,2018,3,1.51,4.0,2.842381,101.4,0.09,2.330950,0.715078,0.000000,0.566667,71380.0,0.0,0.0,251.0,16083.0,-0.992708,-0.900501,-0.1,-0.095727,7.840950,0,1



Shape: (100000, 30)


In [122]:
# VALIDATE MACRO FEATURE MERGE
# __________________________________________________

logger.info("Validating merged macroeconomic feature table...")

# Row Count Validation
# _____________________________________________________

assert len(train_mapping_macro) == len(train_mapping), \
    "Training merge changed the number of rows."

assert len(test_mapping_macro) == len(test_mapping), \
    "Testing merge changed the number of rows."

print("Row count validation passed.")

# Missing Values Validation
# ___________________________________________

metadata_columns = [
    "row_index",
    "issue_d",
    "YearMonth",
    "loan_status"]

macro_columns = [
    col for col in train_mapping_macro.columns
    if col not in metadata_columns]

train_missing = train_mapping_macro[macro_columns].isnull().sum().sum()
test_missing = test_mapping_macro[macro_columns].isnull().sum().sum()

print(f"Training Missing Macro Values : {train_missing}")
print(f"Testing Missing Macro Values  : {test_missing}")

assert train_missing == 0, "Training macro features contain missing values."
assert test_missing == 0, "Testing macro features contain missing values."

print("Missing value validation passed.")


# Duplicate Validation
# ------------------------------------------------------------

print(f"Training Duplicate Rows : {train_mapping_macro.duplicated().sum()}")

print(f"Testing Duplicate Rows  : {test_mapping_macro.duplicated().sum()}")

logger.info("Macro feature merge validation completed successfully.")

Row count validation passed.
Training Missing Macro Values : 0
Testing Missing Macro Values  : 0
Missing value validation passed.
Training Duplicate Rows : 0
Testing Duplicate Rows  : 0


### Extract Macro Feature Columns.

#### Purpose
##### ----------------------
##### create the new feature block that will be added to Matrix C.

In [128]:
# EXTRACT MACRO FEATURE BLOCK
# ______________________________________________

logger.info("Extracting macro feature columns...")

# Columns excluded from modelling

exclude_columns = [
    "row_index",
    "issue_d",
    "YearMonth",
    "loan_status",
    "DATE",
    "Year",
    "Month",
    "MACRO_DATA_AVAILABLE",
    "DATE_AVAILABLE"]

# Select only modelling features

macro_feature_columns = [col for col in train_mapping_macro.columns
    
                         if col not in exclude_columns]

logger.info(f"Number of macro features selected: {len(macro_feature_columns)}")

print("Selected Macro Features:")

for feature in macro_feature_columns:
    
    print(feature)

# Create feature blocks

X_macro_train = train_mapping_macro[macro_feature_columns]

X_macro_test = test_mapping_macro[macro_feature_columns]

print("\nTraining Macro Shape:")
print(X_macro_train.shape)

print("\nTesting Macro Shape:")
print(X_macro_test.shape)

Selected Macro Features:
FEDFUNDS
UNRATE
DGS10
UMCSENT
FEDFUNDS_MoM
CPIAUCSL_YoY
CPIAUCSL_3M_Momentum
GDPC1_Growth
YIELD_CURVE
Articles
LOAN
MORTGAGE
FEDERAL_RESERVE
ECONOMY
Average_Tone
Average_Tone_LAG3
UNRATE_CHANGE
Average_Tone_CHANGE
ECONOMIC_STRESS_INDEX
RECESSION_FLAG
HIGH_RATE_ENVIRONMENT

Training Macro Shape:
(400000, 21)

Testing Macro Shape:
(100000, 21)


#### Convert Macro Features to Sparse Matrix

In [130]:
# CONVERT MACRO FEATURES TO SPARSE MATRIX
# ____________________________________________

logger.info("Converting macro features to sparse matrices...")

# Convert training macro features
# ____________________________________________

X_macro_train_sparse = csr_matrix(X_macro_train.values)

# Convert testing macro features
# ____________________________________________

X_macro_test_sparse = csr_matrix(X_macro_test.values)

# Validate shapes
# ____________________________________________

print("Training Macro Sparse Matrix Shape:")

print(X_macro_train_sparse.shape)

print("\nTesting Macro Sparse Matrix Shape:")

print(X_macro_test_sparse.shape)

print("\nTraining Macro Matrix Type:")

print(type(X_macro_train_sparse))

logger.info("Macro features successfully converted to sparse format.")

Training Macro Sparse Matrix Shape:
(400000, 21)

Testing Macro Sparse Matrix Shape:
(100000, 21)

Training Macro Matrix Type:
<class 'scipy.sparse._csr.csr_matrix'>


In [132]:
# VALIDATE MATRIX C
# _______________________________

assert X_macro_train_sparse.shape[0] == MATRIX_C_TRAIN.shape[0], \
    "Training macro rows do not match Matrix C rows."

assert X_macro_test_sparse.shape[0] == MATRIX_C_TEST.shape[0], \
    "Testing macro rows do not match Matrix C rows."


print("Macro features and Matrix C row alignment confirmed.")

Macro features and Matrix C row alignment confirmed.


### BUILDING MATRIX D (Matrix C + Macro Features)

In [224]:
# BUILD MATRIX D
# ___________________________________

logger.info("Building Matrix D by combining Matrix C with macro features...")

# Combine Training Matrix C + Macro Features
# ______________________________________________

matrix_D_train = hstack([MATRIX_C_TRAIN,X_macro_train_sparse],format="csr")

# Combine Testing Matrix C + Macro Features
# _______________________________________________

matrix_D_test = hstack([MATRIX_C_TEST,X_macro_test_sparse],format="csr")

# Display Matrix D dimensions
# _________________________________________

print("Matrix C Train Shape:")
print(MATRIX_C_TRAIN.shape)

print("\nMacro Train Shape:")
print(X_macro_train_sparse.shape)

print("\nMatrix D Train Shape:")
print(matrix_D_train.shape)

print("\n-----------------------------")

print("\nMatrix C Test Shape:")
print(MATRIX_C_TEST.shape)

print("\nMacro Test Shape:")
print(X_macro_test_sparse.shape)

print("\nMatrix D Test Shape:")
print(matrix_D_test.shape)

logger.info("Matrix D successfully created.")

Matrix C Train Shape:
(400000, 70)

Macro Train Shape:
(400000, 21)

Matrix D Train Shape:
(400000, 91)

-----------------------------

Matrix C Test Shape:
(100000, 70)

Macro Test Shape:
(100000, 21)

Matrix D Test Shape:
(100000, 91)


In [225]:
# VALIDATE MATRIX D
# _____________________________________

logger.info("Validating Matrix D...")

# Row Count Validation
# _____________________________________

assert matrix_D_train.shape[0] == MATRIX_C_TRAIN.shape[0], \
    "Training row count changed after adding macro features."


assert matrix_D_test.shape[0] == MATRIX_C_TEST.shape[0], \
    "Testing row count changed after adding macro features."


print("✅ Row count validation passed.")

# Feature Expansion Validation
# ______________________________________________________________

expected_train_features = (
    MATRIX_C_TRAIN.shape[1]
    +
    X_macro_train_sparse.shape[1])

expected_test_features = (
    MATRIX_C_TEST.shape[1]
    +
    X_macro_test_sparse.shape[1])

assert matrix_D_train.shape[1] == expected_train_features, \
    "Training feature count mismatch."


assert matrix_D_test.shape[1] == expected_test_features, \
    "Testing feature count mismatch."


print("Feature expansion validation passed.")

# Missing Value Check
# ___________________________________________________________

print("\nMatrix D contains:")

print("NaN values: Not applicable (Sparse Matrix)")

print("Sparse format:", matrix_D_train.getformat())

# Sparsity Check
# ________________________________________________________

train_density = (
    matrix_D_train.nnz /
    (matrix_D_train.shape[0] * matrix_D_train.shape[1]))


test_density = (
    matrix_D_test.nnz /
    (matrix_D_test.shape[0] * matrix_D_test.shape[1]))


print("\nMatrix D Density")
print("----------------")
print(f"Training Density: {train_density:.6f}")
print(f"Testing Density : {test_density:.6f}")


logger.info("Matrix D validation completed successfully.")

✅ Row count validation passed.
Feature expansion validation passed.

Matrix D contains:
NaN values: Not applicable (Sparse Matrix)
Sparse format: csr

Matrix D Density
----------------
Training Density: 0.287885
Testing Density : 0.287922


#### CREATE MATRIX D FEATURE NAMES

In [226]:
# Load Matrix C feature names

matrix_C_features_df = pd.read_csv(FEATURE_REPORT_DIR / "combined_feature_names.csv")

matrix_C_feature_names = (matrix_C_features_df["Feature_Name"].tolist())

print("Matrix C Features:", len(matrix_C_feature_names))

# Create Matrix D feature names

matrix_D_feature_names = (matrix_C_feature_names +macro_feature_columns)

print("Matrix D Features:", len(matrix_D_feature_names))

Matrix C Features: 70
Matrix D Features: 91


In [227]:
# validate matrix_D 
# ___________________________________

matrix_D_feature_names = (matrix_C_feature_names + macro_feature_columns)

print("Matrix D Features:", len(matrix_D_feature_names))

print("Matrix D Columns:", matrix_D_train.shape[1])

assert len(matrix_D_feature_names) == matrix_D_train.shape[1]

print("Matrix D feature names validated.")

Matrix D Features: 91
Matrix D Columns: 91
Matrix D feature names validated.


## Save Matrix D

In [228]:
# SAVE MATRIX D
# ________________________________________________

logger.info("Saving Matrix D...")

# Save sparse matrices

sparse.save_npz(FEATURE_STORE_DIR / "matrix_D_train.npz",matrix_D_train)

sparse.save_npz(FEATURE_STORE_DIR / "matrix_D_test.npz",matrix_D_test)

# Save feature names

matrix_D_feature_df = pd.DataFrame({"Feature_ID": range(len(matrix_D_feature_names))
                                    
                                    ,"Feature_Name": matrix_D_feature_names})


matrix_D_feature_df.to_csv(FEATURE_STORE_DIR / "matrix_D_feature_names.csv",index=False)

logger.info("Matrix D saved successfully.")

print("✅ Matrix D export complete.")

✅ Matrix D export complete.


In [217]:
# SAVE TARGET VARIABLES
# _________________________________________________

logger.info("Saving target variables...")

sparse.save_npz("matrix_D_train.npz",matrix_D_train)

sparse.save_npz("matrix_D_test.npz",matrix_D_test)

logger.info("Target variables saved successfully.")